# **NPC Generator**

This file contains individual functions for generating each NPC key.
Each function returns a string or appropriate data type for that key.

## Structure of what we need

- **With random**:
id / gender / age

- **From datasets**:
name / culture / appearance / personality / traits

- **OpenAI request**:
history / portrait / goal / occupation


In [1]:
import uuid
from getpass import getpass
from pathlib import Path
import random
from openai import OpenAI
from typing import Dict, List, Any
import json
import requests
from pprint import pprint

## Helper functions

- Save to Json

In [2]:
def save_npcs_to_json(npcs, filename="NPCs.json"):    
    # Convert single NPC to list if needed
    if isinstance(npcs, dict):
        npcs = [npcs]
    
    # Create output directory if it doesn't exist
    output_dir = Path("../Unity/Echoes of the crowd/Assets/StreamingAssets/NPC")
    
    # Save to file
    output_path = output_dir / filename
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(npcs, f, indent=2, ensure_ascii=False)
    
    print(f"Saved {len(npcs)} NPC(s) to {output_path}")
    return output_path


- Gather the APIKey

In [4]:
api_key = getpass("Enter your OpenAI API key: ")

In [5]:
client = OpenAI(api_key=api_key)

- Call OpenAI

In [6]:
def call_openai_api(content: str, prompt : str, max_tokens : int, temperature : float):
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": content},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )
    # This is a placeholder for the actual OpenAI API call
    return response.choices[0].message.content.strip()


- Call Image creation

In [8]:
def call_image_api(name: str, prompt: str):
    response = client.images.generate(
        model="dall-e-3",  # or "gpt-image-1"
        prompt=prompt,
        size="1024x1024",
        n=1
    )
    
    image_url = response.data[0].url    
    unity_streaming_assets_path = Path("../Unity/Echoes of the crowd/Assets/StreamingAssets")       

    safe_name = name.replace(" ", "_")
    filename = f"portrait_{safe_name}.png"

    file_path = unity_streaming_assets_path / filename
    
    img_response = requests.get(image_url)
    img_response.raise_for_status()

    with open(file_path, 'wb') as f:
        f.write(img_response.content)
        
    if isinstance(file_path, Path):
        return str(file_path.name)  # Return just the filename as string
    else:
        return str(file_path)  # Return as string

## Keys function gathered with Random

**Key: ID**

In [12]:
def generate_id() -> str: 
    return f"npc_{str(uuid.uuid4())[:8]}"

In [26]:
generate_id()

'npc_1ae08dc7'

**Key: Gender**

In [13]:
def generate_gender() -> str:
    return random.choice(["Male", "Female"])

In [27]:
gender = generate_gender()
print(f"Gender: {gender}")

Gender: Female


**Key: Age**

In [14]:
def generate_age() -> int:
    return random.randint(10, 90)

In [15]:
age = generate_age()
print(f"Age: {age}")

Age: 77


## Keys functions gathered from AI Request

**Key: Culture**

In [9]:
def generate_culture() -> str:
# Add variety to culture generation
    regions = [
        "Northern", "Southern", "Eastern", "Western", "Central", "Coastal", "Mountainous", "Desert", "Tropical", "Arctic", 
    ]
    region = random.choice(regions)
    
    time_periods = [
        "ancient", "medieval", "renaissance", "classical", "tribal", "imperial", "nomadic", "settled", "feudal", "mythical"
    ]
    time_period = random.choice(time_periods)
    
    prompt = f"""
    Generate a unique fantasy culture name for an RPG character.
    Focus on {region} regions and {time_period} time periods.
    The culture should be inspired by real-world cultures but with a fantasy twist.
    Consider geographical regions, historical periods, and cultural traditions.
    
    Examples: Northern European, East Asian, Mediterranean, Middle Eastern, West African, Nordic, Slavic, Celtic, Arabian, Persian, Indian, Mongolian, etc.
    
    Make it unique and avoid common or overused culture names.
    Consider environmental factors, historical influences, and cultural evolution.
    
    Return only the culture name, no additional text or explanation.
    Maximum of 3 words
    """
    content = """ 
        You are a creative writer and historiator helping to generate NPC backgrounds and recovering cultures for a fantasy game. Provide concise, creative responses.
    """

    return call_openai_api(content, prompt, max_tokens=50, temperature=0.8)

In [16]:
culture = generate_culture()
print(f"Culture: {culture}")

Culture: Frostborne Clans


**Key: Name**

In [ ]:
def generate_name(culture: str, gender: str) -> str:
    name_styles = [
        "traditional", "modern", "ancient", "noble", "common", "exotic", "regional", "historical"
    ]
    name_style = random.choice(name_styles)
    
    social_classes = [
        "peasant", "merchant", "noble", "warrior", "scholar", "artisan", "mystic", "wanderer"
    ]
    social_class = random.choice(social_classes)
    
    prompt = f"""
    Generate a realistic full name (first name and last name) for a fantasy RPG {gender} character from {culture} culture.
    The name should be {name_style} and appropriate for a {social_class} in a medieval/fantasy setting.
    Consider typical {culture} naming conventions, cultural influences, and historical naming practices.
    
    Make the name unique and distinctive, avoiding common or overused names.
    Consider {culture} linguistic patterns, traditional naming structures, and cultural significance.
    
    Return only the name, no additional text or explanation.
    Example format: "Firstname Lastname"
    """

    content = """ 
        You are a creative writer and historiator helping to generate NPC backgrounds and recovering cultures for a fantasy game. Provide concise, creative responses.
    """

    return call_openai_api(content, prompt, max_tokens=50, temperature=0.8)

In [23]:
culture = generate_culture()
gender = generate_gender()
name = generate_name(culture, gender)

print(f"Culture: {culture}")
print(f"Gender: {gender}")
print(f"Name: {name}")

Culture: Valenheim Coterie
Gender: Female
Name: Aricel Thalorin


**Key: appearance**

In [27]:
def generate_appearance(age: int, gender:str, culture: str) -> Dict[str, Any]:    
    lifestyles = ["active", "sedentary", "athletic", "scholarly", "artistic", "military", "merchant", "noble"]
    lifestyle = random.choice(lifestyles)
    
    prompt = f"""
    Generate realistic appearance details for a {age} {gender} from {culture} culture with a {lifestyle} lifestyle.
    Consider the typical physical characteristics, traditional features, and cultural influences of {culture} people.
    
    You must return the response with exactly this structure:
    {{
        "hair_color": "specific color name",
        "eye_color": "specific color name", 
        "height_cm": integer number between 150-200,
        "build": "specific build description"
    }}
    
    Make the appearance culturally appropriate and realistic for {culture} heritage.
    Use specific, descriptive terms for colors and builds.
    Consider how {lifestyle} lifestyle and {age} would affect their physical appearance.
    
    Respond with valid JSON only, no additional text.
    """

    content = """ 
        You are a creative writer and historiator helping to generate NPC backgrounds and recovering cultures for a fantasy game. Provide concise, creative responses.
    """

    response = call_openai_api(content, prompt, max_tokens=100, temperature=0.8)
    
    # Parse the JSON response
    appearance_dict = json.loads(response)
    return appearance_dict

In [28]:
appearance = generate_appearance(age, gender, culture)
print(f"Appearance: {appearance}")

Appearance: {'hair_color': 'silver-grey', 'eye_color': 'emerald green', 'height_cm': 165, 'build': 'slender with a slightly hunched posture, indicative of years spent poring over tomes'}


**Key: personality**

In [29]:
def generate_personality(culture: str) -> Dict[str, float]:
    life_experiences = [
        "adventurous", "peaceful", "challenging", "privileged", "humble", "tragic", "successful", "struggling"
    ]
    life_experience = random.choice(life_experiences)
    
    social_roles = [
        "leader", "follower", "outsider", "insider", "mediator", "rebel", "traditionalist", "innovator"
    ]
    social_role = random.choice(social_roles)
    
    prompt = f"""
    Generate personality traits using the Big Five personality model for a person from {culture} culture.
    This person has lived a {life_experience} life and tends to be a {social_role} in their community.
    Consider how {culture} cultural values, traditions, and social norms might influence personality development.
    
    You must return the response as valid JSON with exactly this structure:
    {{
        "openness": float number between 0.0 and 1.0,
        "conscientiousness": float number between 0.0 and 1.0,
        "extraversion": float number between 0.0 and 1.0,
        "agreeableness": float number between 0.0 and 1.0,
        "neuroticism": float number between 0.0 and 1.0
    }}
    
    Create a realistic and balanced personality profile that reflects {culture} cultural influences.
    Consider how their {life_experience} experiences and {social_role} role would shape their personality.
    Use decimal numbers (e.g., 0.75, 0.32) for all values.
    
    Respond with valid JSON only, no additional text or markdown formatting.
    """
    
    content = """ 
        You are a creative writer and historiator helping to generate NPC backgrounds and recovering cultures for a fantasy game. Provide concise, creative responses.
    """
    
    response = call_openai_api(content, prompt, max_tokens=100, temperature=0.8)
    
    # Clean the response to remove markdown formatting
    cleaned_response = response.strip()
    if cleaned_response.startswith('```json'):
        cleaned_response = cleaned_response[7:]  # Remove '```json'
    if cleaned_response.endswith('```'):
        cleaned_response = cleaned_response[:-3]  # Remove '```'
    cleaned_response = cleaned_response.strip()
    
    # Parse the cleaned JSON
    personality_dict = json.loads(cleaned_response)
    return personality_dict

In [30]:
personality = generate_personality(culture)
print(f"Personality: {personality}")

Personality: {'openness': 0.45, 'conscientiousness': 0.75, 'extraversion': 0.4, 'agreeableness': 0.85, 'neuroticism': 0.3}


**Key: traits**

In [31]:
def generate_traits(culture: str, personality: Dict[str, float]) -> List[str]:
    trait_categories = [
        "personal qualities", "social behaviors", "work habits", "emotional responses", "moral values", "intellectual characteristics"
    ]
    trait_category = random.choice(trait_categories)
    
    prompt = f"""
    Based on this personality profile: {personality}
    And considering the cultural background of {culture}
    
    Generate 3-4 character traits focusing on {trait_category} that would naturally emerge from this personality and cultural upbringing.
    Consider how {culture} cultural values, traditions, and social expectations might shape these traits.
    
    You must return the response as valid JSON with exactly this structure:
    {{
        "traits": ["trait1", "trait2", "trait3", "trait4"]
    }}
    
    Make the traits realistic and consistent with both the personality scores and {culture} cultural context.
    Use specific, descriptive trait words that reflect {trait_category}.
    Avoid generic or overused trait words.
    
    Respond with valid JSON only, no additional text.
    """
    
    content = """ 
        You are a creative writer and historiator helping to generate NPC backgrounds and recovering cultures for a fantasy game. Provide concise, creative responses.
    """
    
    response = call_openai_api(content, prompt, max_tokens=100, temperature=0.8)
    
    # Parse the JSON string and extract the traits list
    traits_data = json.loads(response)
    traits_list = traits_data.get('traits', [])
    return traits_list

In [32]:
traits = generate_traits(culture, personality)
print(f"Traits: {traits}")

Traits: ['pragmatic thinker', 'empathetic strategist', 'methodical analyst', 'culturally inquisitive']


**Key: occupation**

In [33]:
def generate_occupation(npc_data: Dict[str, Any]) -> str:
    culture = npc_data.get('culture', 'Unknown')
    
    # Add variety to occupation generation
    skill_levels = [
        "apprentice", "journeyman", "master", "expert", "novice", "seasoned", "veteran", "specialist"
    ]
    skill_level = random.choice(skill_levels)
    
    work_environments = [
        "urban setting", "rural village", "coastal town", "mountain settlement", "desert oasis", "forest community", "trading post", "religious center"
    ]
    work_environment = random.choice(work_environments)
    
    prompt = f"""
    Suggest an appropriate occupation for an NPC from {culture} culture with the following characteristics:
    - Name: {npc_data.get('name', 'Unknown')}
    - Age: {npc_data.get('age', 'Unknown')}
    - Culture: {culture}
    - Personality: {npc_data.get('personality', {})}
    - Traits: {npc_data.get('traits', [])}
    - Brief History: {npc_data.get('brief_history', 'Unknown')}
    - Goal: {npc_data.get('goal', 'Unknown')}
    
    Consider them as a {skill_level} working in a {work_environment}.
    Suggest a single occupation that fits their background, personality, and {culture} cultural context.
    Consider traditional {culture} professions, cultural values, and social roles.
    Make it suitable for a fantasy RPG setting.
    
    Focus on occupations that would be common or respected within {culture} society and reflect {culture} cultural traditions.
    Make the occupation specific and avoid generic or overused fantasy professions.
    Return only the occupation name, no additional text.
    Return maximum 3 words.
    """
    
    content = """ 
        You are a creative writer and historiator helping to generate NPC backgrounds and recovering cultures for a fantasy game. Provide concise, creative responses.
    """
    
    return call_openai_api(content, prompt, max_tokens=100, temperature=0.8)

In [34]:
npc_data = {
        'id': generate_id(),
        'name': name,
        'gender': gender,
        'age': age,
        'culture': culture,
        'appearance': appearance,
        'personality': personality,
        'traits': traits
    }

occupation = generate_occupation(npc_data)
print(f"Occupation: {occupation}")

Occupation: Oasis Mediator Sage


**Key: goal**

In [35]:
def generate_goal(npc_data: Dict[str, Any]) -> str:
    culture = npc_data.get('culture', 'Unknown')
    
    # Add variety to goal generation
    goal_categories = [
        "personal achievement", "family honor", "community service", "knowledge seeking", "wealth accumulation", "spiritual enlightenment", "social recognition", "creative expression"
    ]
    goal_category = random.choice(goal_categories)
    
    timeframes = [
        "immediate future", "next few years", "lifetime ambition", "before retirement", "before marriage", "before the next season", "before the festival", "before winter"
    ]
    timeframe = random.choice(timeframes)
    
    prompt = f"""
    Create a personal goal for an NPC from {culture} culture with the following characteristics:
    - Name: {npc_data.get('name', 'Unknown')}
    - Age: {npc_data.get('age', 'Unknown')}
    - Culture: {culture}
    - Personality: {npc_data.get('personality', {})}
    - Traits: {npc_data.get('traits', [])}
    - Brief History: {npc_data.get('brief_history', 'Unknown')}
    
    Focus on {goal_category} as their primary goal, with a timeframe of {timeframe}.
    Write a single sentence describing their primary life goal or ambition.
    Consider how {culture} cultural values, traditions, and social expectations might influence their aspirations.
    Make it compelling and suitable for a fantasy RPG setting.
    
    Focus on goals that would be meaningful within {culture} society and reflect {culture} cultural priorities.
    Make the goal specific and personal, avoiding generic or clichéd ambitions.
    Maximum of 20 tokens
    """
    
    content = """ 
        You are a creative writer and historiator helping to generate NPC backgrounds and recovering cultures for a fantasy game. Provide concise, creative responses.
    """

    response = call_openai_api(content, prompt, max_tokens=100, temperature=0.8)
    
    return response

**Key: history**

In [37]:
def generate_history(npc_data: Dict[str, Any]) -> str:
    culture = npc_data.get('culture', 'Unknown')
    
    # Add variety to history generation
    story_themes = [
        "overcoming adversity", "pursuing dreams", "family legacy", "personal growth", "cultural conflict", "social mobility", "spiritual journey", "professional achievement", "dramatic change"
    ]
    story_theme = random.choice(story_themes)
    
    current_situations = [
        "seeking new opportunities", "returning home", "starting fresh", "facing challenges", "celebrating success", "mourning loss", "preparing for change", "finding purpose", "revenge"
    ]
    current_situation = random.choice(current_situations)
    
    prompt = f"""
    Create a brief background story for an NPC from {culture} culture with the following characteristics:
    - Name: {npc_data.get('name', 'Unknown')}
    - Gender: {npc_data.get('gender', 'Unknown')}
    - Age: {npc_data.get('age', 'Unknown')}
    - Culture: {culture}
    - Appearance: {npc_data.get('appearance', {})}
    - Personality: {npc_data.get('personality', {})}
    - Traits: {npc_data.get('traits', [])}
    
    Focus the story on {story_theme} and their current situation of {current_situation}.
    Write background story that explains their upbringing in {culture} society and current situation.
    Emphasize how their {culture} cultural background has shaped their life experiences and current circumstances.
    Make it engaging and suitable for a fantasy RPG setting.
    
    Focus on cultural elements, traditions, and how {culture} society influenced their development.
    Make the story unique and avoid generic or clichéd backgrounds.
    Maximum of 100 tokens
    """
    
    content = """ 
        You are a creative writer and historiator helping to generate NPC backgrounds and recovering cultures for a fantasy game. Provide concise, creative responses.
    """

    response = call_openai_api(content, prompt, max_tokens=150, temperature=0.8)

    return response

**Key: portrait**

In [38]:
def generate_portrait(name: str, gender: str,culture: str, age: int, appearance) -> str:
    if isinstance(appearance, str):
        appearance_dict = json.loads(appearance)
    else:
        appearance_dict = appearance
    
    art_styles = [
        "realistic", "fantasy", "medieval", "renaissance", "modern", "classical", "romantic", "impressionist"
    ]
    art_style = random.choice(art_styles)
    
    portrait_types = [
        "head and shoulders", "full body", "three-quarter view", "bust", "character concept"
    ]
    portrait_type = random.choice(portrait_types)
    
    prompt = f"""
    Generate a {art_style} {portrait_type} portrait of a {age}-year-old {gender} from {culture} culture with:
    - Hair: {appearance_dict.get('hair_color', 'unknown')}
    - Eyes: {appearance_dict.get('eye_color', 'unknown')}
    - Build: {appearance_dict.get('build', 'unknown')}
    - Height: {appearance_dict.get('height_cm', 'unknown')}cm
    
    Style: Fantasy RPG character portrait, detailed, professional, {art_style} art style.
    Emphasize {culture} cultural features, traditional clothing elements, and cultural authenticity.
    Include {culture} cultural symbols, traditional attire, and ethnic features appropriate for {culture} heritage.
    Make the portrait unique and distinctive, avoiding generic or stereotypical representations.
    ### IMPORTANT ###
    Follow this rules:
     - Respect the gender of the character
     - No texts
     - No letters
     - Only show one character in image
     - No watermarks
     - No logos
    """
    
    return call_image_api(name, prompt)

In [88]:
print(appearance)

{
    "hair_color": "frosty white",
    "eye_color": "deep azure",
    "height_cm": 168,
    "build": "slender with a slight stoop from years of reading and study"
}


In [39]:
portrait = generate_portrait(name, gender, culture, age, appearance)
print(f"Portrait saved at: {portrait}")

Portrait saved at: portrait_Aricel_Thalorin.png


## Final function -- Create NPC

In [42]:
def generate_npc():

    id = generate_id()
    gender = generate_gender()
    age = generate_age()
    culture = generate_culture()
    name = generate_name(culture,gender)

    appearance_response = generate_appearance(age, gender, culture)
    if isinstance(appearance_response, str):
        appearance = json.loads(appearance_response)
    else:
        appearance = appearance_response

    personality_response = generate_personality(culture)
    if isinstance(personality_response, str):
        personality = json.loads(personality_response)
    else:
        personality = personality_response   

    traits_response = generate_traits(culture, personality)
    if isinstance(traits_response, str):
        traits_data = json.loads(traits_response)
        traits = traits_data.get('traits', [])
    else:
        traits = traits_response
    
    # Occupation
    npc_data = {
        'id': id,
        'name': name,
        'gender': gender,
        'age': age,
        'culture': culture,
        'appearance': appearance,
        'personality': personality,
        'traits': traits
    }

    brief_history = generate_history(npc_data)
    portrait = generate_portrait(name, gender, culture, age, appearance)

    npc_data.update({
        'brief_history': brief_history,
        'portrait': portrait
    })
    
    goal = generate_goal(npc_data)
    occupation = generate_occupation(npc_data)
    
    complete_npc = {
        'id': id,
        'name': name,
        'gender': gender,
        'age': age,
        'culture': culture,
        'appearance': appearance,
        'personality': personality,
        'traits': traits,
        'brief_history': brief_history,
        'portrait': portrait,
        'goal': goal,
        'occupation': occupation
    }

    return complete_npc

In [43]:
npc = generate_npc()
print("NPC generated successfully")
print(f"Portrait field: {npc.get('portrait')}")
print(f"Goal field: {npc.get('goal')}")
print(f"Occupation field: {npc.get('occupation')}")
save_npcs_to_json(npc)


NPC generated successfully
Portrait field: portrait_Alaric_Searhaven.png
Goal field: Alaric aims to rebuild the Searhaven legacy by uniting the fractured Corsair clans before winter, restoring honor through shared voyages and tales of valor.
Occupation field: Faithful Tidekeeper
Saved 1 NPC(s) to ..\Unity\Echoes of the crowd\Assets\StreamingAssets\NPC\NPCs.json


WindowsPath('../Unity/Echoes of the crowd/Assets/StreamingAssets/NPC/NPCs.json')

In [44]:
def generate_multiple_npcs(count):
    npcs = []
    
    for i in range(count):
        try:
            npc = generate_npc()
            npcs.append(npc)
            print(f"✓ Generated: Name-> {npc['name']}, Gender-> {npc['gender']}, Culture-> ({npc['culture']})")
        except Exception as e:
            print(f"✗ Error generating NPC {i+1}: {e}")
            continue
    
    # Save to JSON
    save_npcs_to_json(npcs)
    
    return npcs

In [46]:
generate_multiple_npcs(10)

✓ Generated: Name-> Maris Tideweaver, Gender-> Male, Culture-> (Seashell Elysium)
✓ Generated: Name-> Elysara Veyril, Gender-> Male, Culture-> (Silkwind Archons)
✓ Generated: Name-> Eldaer Thorneweaver, Gender-> Female, Culture-> (Fjordwyn Kinfolk)
✓ Generated: Name-> Elyndor Telvashan, Gender-> Female, Culture-> (Valekaran Dominion)
✓ Generated: Name-> Zariel Nyxoran, Gender-> Female, Culture-> (Celestial Omenkin)
✓ Generated: Name-> Kalthar Sirothar, Gender-> Male, Culture-> (Zarathian Dunesfolk)
✓ Generated: Name-> Elandor Grimsworn, Gender-> Male, Culture-> (Fjordalith Kinship)
✓ Generated: Name-> Thalira Frostweaver, Gender-> Male, Culture-> (Frostmarrow Syndicate)
✓ Generated: Name-> Lirael Nyssarion, Gender-> Male, Culture-> (Tideborne Maraliths)
✓ Generated: Name-> Eldrin Frostwhisper, Gender-> Female, Culture-> (Frostborne Thalassans)
Saved 10 NPC(s) to ..\Unity\Echoes of the crowd\Assets\StreamingAssets\NPC\NPCs.json


[{'id': 'npc_0e35f0dd',
  'name': 'Maris Tideweaver',
  'gender': 'Male',
  'age': 63,
  'culture': 'Seashell Elysium',
  'appearance': {'hair_color': 'salt-and-pepper grey',
   'eye_color': 'seafoam green',
   'height_cm': 178,
   'build': 'muscular and well-defined, with signs of wear on joints from years of service'},
  'personality': {'openness': 0.45,
   'conscientiousness': 0.6,
   'extraversion': 0.4,
   'agreeableness': 0.75,
   'neuroticism': 0.7},
  'traits': ['empathetic communicator',
   'cautiously sociable',
   'attentive collaborator',
   'reflective peacemaker'],
  'brief_history': 'Maris Tideweaver, born amidst the vibrant coral reefs of Seashell Elysium, spent his youth sculpting intricate shells into art. His salt-and-pepper hair reflects years spent beneath the sun and surf, while his seafoam eyes mirror the depths of his dreams—dreams of uniting the disparate clans through shared cultural stories. At 63, weary joints testify to his life as a peacemaker, yet his hea